<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/Test/2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step.
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file.
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:**


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:**

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters())
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:**

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:**

In [18]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple


In [19]:
# Create the directory if it doesn't exist
!mkdir -p /opt/cocoapi/annotations/

# Download the annotations zip file
!wget http://images.cocodataset.org/annotations/annotations_trainval2014.zip -P /opt/cocoapi/annotations/
!unzip /opt/cocoapi/annotations/annotations_trainval2014.zip -d /opt/cocoapi/annotations/
!rm /opt/cocoapi/annotations/annotations_trainval2014.zip

# Move the files from the nested 'annotations' directory to the parent 'annotations' directory
!mv /opt/cocoapi/annotations/annotations/* /opt/cocoapi/annotations/
!rmdir /opt/cocoapi/annotations/annotations/

--2026-03-12 18:54:43--  http://images.cocodataset.org/annotations/annotations_trainval2014.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.185.77, 52.217.160.185, 54.231.224.201, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.185.77|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252872794 (241M) [application/zip]
Saving to: ‘/opt/cocoapi/annotations/annotations_trainval2014.zip’

annotations_trainva 100%[===================>] 241.16M  51.1MB/s    in 5.3s    

2026-03-12 18:54:49 (45.8 MB/s) - ‘/opt/cocoapi/annotations/annotations_trainval2014.zip’ saved [252872794/252872794]

Archive:  /opt/cocoapi/annotations/annotations_trainval2014.zip
  inflating: /opt/cocoapi/annotations/annotations/instances_train2014.json  
  inflating: /opt/cocoapi/annotations/annotations/instances_val2014.json  
  inflating: /opt/cocoapi/annotations/annotations/person_keypoints_train2014.json  
  inflating: /opt/cocoapi/annotations/anno

In [20]:
import os, re, io, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
from PIL import Image
import matplotlib.pyplot as plt

try:
    # Import BLEU score for evaluation if NLTK is available
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    nltk_ok = True
except Exception:
    nltk_ok = False

SEED = 42
random.seed(SEED) # Set random seed for Python's random module
np.random.seed(SEED) # Set random seed for NumPy
torch.manual_seed(SEED) # Set random seed for PyTorch CPU operations
torch.cuda.manual_seed_all(SEED) # Set random seed for PyTorch CUDA (GPU) operations

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Determine computing device (GPU if available, else CPU)
print('Device:', device)

SPECIAL_TOKENS = {'pad':'', 'bos':'', 'eos':'', 'unk':''} # Define special tokens for vocabulary

CFG = {
    'dataset': 'COCO',          # 'Flickr8k' or 'COCO'
    'data_root': '/data',

    # COCO expected structure:
    #   ./data/COCO/train2017/*.jpg
    #   ./data/COCO/val2017/*.jpg
    #   ./data/COCO/annotations/captions_train2017.json
    #   ./data/COCO/annotations/captions_val2017.json

    'min_freq': 5, # Minimum frequency for a word to be included in the vocabulary
    'max_len': 20, # Maximum caption length
    'batch_size': 1,
    'num_workers': 2,

    # Model & training configurations
    'use_spatial_attention': True,   # Toggle between global and spatial attention
    'encoder_cnn': 'resnet50',
    'embed_dim': 256,
    'hidden_dim': 512,
    'attention_dim': 256,
    'dropout': 0.3,

    'epochs': 10,            # Number of training epochs (increase for real training)
    'lr': 3e-4, # Learning rate
    'clip': 1.0, # Gradient clipping value
    'teacher_forcing': 0.5, # Teacher forcing ratio for decoder training

    # Decoding strategy
    'decode': 'beam',       # 'greedy' or 'beam' search decoding
    'beam_size': 3,         # Beam width when decode='beam' (inference only)

    'save_dir': '/content/drive/MyDrive/image_captioning_checkpoints', # Directory to save model checkpoints
    'exp_name': 'captioning_unified', # Experiment name for saving files
    'fp16': True, # Enable mixed precision training
}

os.makedirs(CFG['save_dir'], exist_ok=True) # Create the save directory if it doesn't exist

Device: cuda


In [21]:
import os
if not os.path.exists('data_loader.py'):
    !wget https://raw.githubusercontent.com/pradhapmoorthi/CVND/Test/data_loader.py
if not os.path.exists('vocabulary.py'):
    !wget https://raw.githubusercontent.com/pradhapmoorthi/CVND/Test/vocabulary.py
if not os.path.exists('model.py'):
    !wget https://raw.githubusercontent.com/pradhapmoorthi/CVND/Test/model.py

In [34]:
import nltk
nltk.download('punkt_tab')

from model import EncoderCNN, DecoderRNN
from vocabulary import Vocabulary # Assuming Vocabulary class is in vocabulary.py
import os

# Path to the annotation file
caption_path = os.path.join('/opt/cocoapi/annotations/', 'captions_train2014.json')

# Build vocabulary (assuming CFG and SPECIAL_TOKENS are defined in a previous cell)
# The Vocabulary.__init__ method automatically builds the vocabulary if vocab_from_file is False (default).
# It expects 'vocab_threshold' and 'annotations_file' in its constructor.
# Also, set the special token words to match the notebook's CFG.
vocab = Vocabulary(
    vocab_threshold=CFG['min_freq'],
    annotations_file=caption_path,
    start_word=SPECIAL_TOKENS['bos'],
    end_word=SPECIAL_TOKENS['eos'],
    unk_word=SPECIAL_TOKENS['unk']
)

# Instantiate Encoder and Decoder. Note: 'vocab' must be defined before this cell is executed.
encoder = EncoderCNN(CFG['embed_dim']).to(device) # Spatial encoder
decoder = DecoderRNN(
        vocab_size=len(vocab),
        embed_size=CFG['embed_dim'],
        hidden_size=CFG['hidden_dim'],
        dropout=CFG['dropout']).to(device) # Spatial decoder

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


loading annotations into memory...
Done (t=1.09s)
creating index...
index created!
[0/414113] Tokenizing captions...
[100000/414113] Tokenizing captions...
[200000/414113] Tokenizing captions...
[300000/414113] Tokenizing captions...
[400000/414113] Tokenizing captions...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

In [35]:
def train_one_epoch(epoch):
    """
    Conducts a single training epoch for the image captioning model.
    Args:
        epoch (int): The current epoch number.
    Returns:
        float: The average training loss for the epoch.
    """
    encoder.train(); decoder.train() # Set models to training mode
    total = 0.0 # Initialize total loss for the epoch
    step_group_start_time = time.time() # Initialize timer for step groups
    for i,(imgs,caps,lens) in enumerate(train_loader):
        imgs, caps = imgs.to(device), caps.to(device) # Move data to appropriate device
        optimizer.zero_grad() # Clear gradients
        with torch.cuda.amp.autocast(enabled=CFG['fp16']):
            if CFG['use_spatial_attention']:
                feats, _ = encoder(imgs) # Encode images with spatial encoder
                logits = decoder(feats, caps) # Decode captions with spatial decoder
            else:
                feat = encoder(imgs) # Encode images with global encoder
                logits = decoder(feat, caps) # Decode captions with global decoder
            targets = caps[:,1:logits.size(1)+1] # Prepare target captions (shifted by one for prediction)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1)) # Calculate loss
        scaler.scale(loss).backward() # Scale loss and perform backward pass
        nn.utils.clip_grad_norm_(params, CFG['clip']) # Clip gradients to prevent exploding gradients
        scaler.step(optimizer) # Update optimizer weights
        scaler.update() # Update the scaler for the next iteration
        total += loss.item() # Accumulate total loss
        if (i+1)%50==0:
            elapsed_time = time.time() - step_group_start_time # Calculate elapsed time for 50 steps
            print(f"epoch {epoch} step {i+1}/{len(train_loader)} loss {total/(i+1):.4f} (last 50 steps took {elapsed_time:.2f}s)")
            step_group_start_time = time.time() # Reset timer for next 50 steps
    return total/max(1,len(train_loader)) # Return average loss for the epoch

In [36]:
def evaluate_bleu(sample_limit=1000):
    """
    Evaluates the model's performance on the validation set using the BLEU-4 metric.
    Args:
        sample_limit (int): The maximum number of samples from the validation set to evaluate.
    Returns:
        float: The calculated BLEU-4 score.
    """
    encoder.eval(); decoder.eval() # Set models to evaluation mode
    refs, hyps = [], [] # Lists to store reference and hypothesis captions
    with torch.no_grad(): # Disable gradient calculations during evaluation
        count=0
        for imgs, caps, lens in val_loader:
            imgs = imgs.to(device)
            if CFG['use_spatial_attention']:
                feats,_ = encoder(imgs) # Encode images
                # Decode captions using greedy search for spatial attention
                out_ids,_ = decoder.greedy_decode(feats, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])
            else:
                feat = encoder(imgs) # Encode images
                # Decode captions using greedy search for global attention
                out_ids = decoder.greedy_decode(feat, vocab.stoi[SPECIAL_TOKENS['bos']], vocab.stoi[SPECIAL_TOKENS['eos']], max_len=CFG['max_len'])[0]
            for b in range(len(out_ids)):
                tgt_ids = caps[b].tolist() # Convert target caption IDs to list
                # strip bos/eos/pad tokens from reference caption
                try: bos = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['bos']])
                except ValueError: bos=0
                eos = tgt_ids.index(vocab.stoi[SPECIAL_TOKENS['eos']]) if vocab.stoi[SPECIAL_TOKENS['eos']] in tgt_ids else len(tgt_ids)
                ref = vocab.denumericalize(tgt_ids[bos+1:eos]) # Denumericalize reference caption
                hyp = vocab.denumericalize(out_ids[b]) # Denumericalize hypothesized caption
                if ref and hyp:
                    refs.append([ref]) # Add reference caption
                    hyps.append(hyp) # Add hypothesized caption
            count += len(out_ids)
            if count>=sample_limit: break # Break if sample limit is reached
    if nltk_ok and hyps:
        smoothie = SmoothingFunction().method4 # Smoothing function for BLEU score calculation
        return corpus_bleu(refs, hyps, smoothing_function=smoothie) # Calculate BLEU-4 score
    return 0.0 # Return 0 if NLTK is not available or no hypotheses


In [43]:
import torch.optim as optim
from torch.cuda.amp import GradScaler
from data_loader import get_loader # Removed CocoDataset import

# Image transformations
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Build data loaders
# Removed train_dataset and val_dataset instantiation as get_loader handles dataset creation

train_loader = get_loader(
    transform=transform_train,
    mode='train',
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers']
)

val_loader = get_loader(
    transform=transform_val,
    mode='test',
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers']
)

# Initialize loss function, optimizer, and scaler
criterion = nn.CrossEntropyLoss().to(device)

# Specify learnable parameters for the optimizer
params = list(decoder.parameters()) + list(encoder.embed.parameters())
# If you want to train all encoder parameters, use:
# params = list(decoder.parameters()) + list(encoder.parameters())

optimizer = optim.Adam(params, lr=CFG['lr'])
scaler = GradScaler(enabled=CFG['fp16'])


# Original training loop from QcLTn_pLW-BO
best_bleu = 0.0 # Initialize best BLEU score
patience = 5 # Number of epochs to wait for improvement before early stopping
patience_counter = 0 # Counter for patience

# Ensure the save directory for checkpoints exists
os.makedirs(CFG['save_dir'], exist_ok=True)
ckpt_path = Path(CFG['save_dir']) / f"{CFG['exp_name']}.pt"

overall_start_time = time.time() # Start overall training timer

for epoch in range(1, CFG['epochs']+1):
    epoch_start_time = time.time() # Start epoch timer
    tr = train_one_epoch(epoch) # Train for one epoch
    bl = evaluate_bleu(sample_limit=1000) # Evaluate BLEU-4 score
    epoch_duration = time.time() - epoch_start_time # Calculate epoch duration
    print(f"Epoch {epoch}: loss={tr:.4f} BLEU-4={bl:.4f} (duration: {epoch_duration:.2f}s)")

    if bl > best_bleu:
        best_bleu = bl # Update best BLEU score
        patience_counter = 0 # Reset patience counter
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'cfg': CFG}, ckpt_path) # Save best model
        print('Saved best model to ->', ckpt_path)
    else:
        patience_counter += 1 # Increment patience counter
        print(f"BLEU-4 did not improve. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break # Trigger early stopping

total_training_duration = time.time() - overall_start_time # Calculate total training duration
print(f"Total training duration: {total_training_duration:.2f}s")

Vocabulary successfully loaded from vocab.pkl file!
loading annotations into memory...
Done (t=0.65s)
creating index...
index created!
Obtaining caption lengths...


100%|██████████| 414113/414113 [00:25<00:00, 16521.56it/s]


AssertionError: Please change batch_size to 1 if testing your model.

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here.

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.

In [ ]:
# (Optional) TODO: Validate your model.